# Social graphs: fetch and process every dataset

Downloads every SNAP social dataset into Drive once (`DATA_ROOT/social/<source>/raw/`), parses it into the
full cached graph (`graph.pt`), and optionally builds the S2 subgraph caches (`processed/<hash>.pt`).
Every step is skipped when its file exists, so rerunning this notebook is cheap.

**A CPU runtime is fine.** The training notebooks fetch their own dataset when it is missing, so this
notebook is optional; it is useful to prepare everything in one go (Higgs is large: allow time).

## 1. Settings (edit)

`BRANCH` defaults to `main`; set it to another branch to run that code instead.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training

## 2. Runtime check (a CPU runtime is fine here)

In [ ]:
!nvidia-smi || echo 'no GPU: fine for this notebook'

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

## 7. Datasets to fetch (edit)

In [ ]:
# EDIT: drop names you do not need. The musae languages beyond 'twitch-de' are optional.
DATASETS = [
    'twitch-gamers',
    'twitch-de', 'twitch-en', 'twitch-es', 'twitch-fr', 'twitch-pt', 'twitch-ru',
    'reddit-body',
    'higgs-follow', 'higgs-retweet', 'higgs-reply', 'higgs-mention',
]
BUILD_S2_CACHES = True   # also build the subgraph caches of configs/social/<dataset>.yaml

try:
    from gnn_mech.social import SOCIAL_DATASETS
    unknown = [d for d in DATASETS if d not in SOCIAL_DATASETS]
    print('known datasets:', sorted(SOCIAL_DATASETS))
    if unknown:
        print('[warn] not in SOCIAL_DATASETS:', unknown)
except Exception as e:
    print(f'[warn] gnn_mech.social not importable: {e!r}')

## 8. Download and parse (skips what exists)

In [ ]:
import time
from gnn_mech.social import sources

fetched = {}
for name in DATASETS:
    t0 = time.time()
    try:
        fetched[name] = sources.fetch(name, root=DATA_ROOT)
        print(f'[ok]   {name:15s} {time.time() - t0:7.1f} s')
    except Exception as e:
        fetched[name] = None
        print(f'[fail] {name:15s} {e!r}')

## 9. Sizes on Drive

In [ ]:
!du -sh {DATA_ROOT}/social/* 2>/dev/null || echo 'nothing downloaded yet'

In [ ]:
import pandas as pd


def graph_summary(obj):
    # nodes / edges / feature width of whatever fetch returned, if it looks like a PyG Data.
    try:
        return {'nodes': int(obj.num_nodes), 'edges': int(obj.edge_index.size(1)),
                'feat_dim': int(obj.x.size(1)) if getattr(obj, 'x', None) is not None else 0}
    except Exception:
        return {}


display(pd.DataFrame([{'dataset': n, **graph_summary(o)} for n, o in fetched.items()]))

## 10. Build the S2 subgraph caches on Drive (optional, can take a while)

In [ ]:
import gc
from gnn_mech.config import load_config
from gnn_mech.data import load_dataset

S2_DATASETS = ['twitch-gamers', 'reddit-body', 'twitch-de', 'higgs-follow']
if BUILD_S2_CACHES:
    for name in S2_DATASETS:
        path = f'configs/social/{name}.yaml'
        if name not in DATASETS or not os.path.exists(path):
            print(f'[skip] {name}: not selected or {path} missing')
            continue
        t0 = time.time()
        try:
            c = load_config(path, [f'out_dir={OUT_DIR}', f'data.root={DATA_ROOT}'])
            tr, va, te, info = load_dataset(c.data)
            print(f'[ok]   {name:15s} {len(tr)} / {len(va)} / {len(te)} subgraphs, task {info.task}, '
                  f'{time.time() - t0:.0f} s')
            del tr, va, te
        except Exception as e:
            print(f'[fail] {name:15s} {e!r}')
        gc.collect()

## 11. S0 structure statistics (if `gnn_mech.social.structure` provides a summary)

In [ ]:
try:
    from gnn_mech.social import structure
    fn = next((getattr(structure, n) for n in ('structure_table', 'summary_table', 'summarize_all',
                                               'dataset_summary', 'summarize', 'summary')
               if callable(getattr(structure, n, None))), None)
    if fn is None:
        print('no summary function found; public names:', [n for n in dir(structure) if not n.startswith('_')])
    else:
        try:
            table = fn(DATASETS, root=DATA_ROOT)
        except TypeError:
            table = pd.DataFrame([{'dataset': n, **dict(fn(n, root=DATA_ROOT))} for n in DATASETS])
        table = table if isinstance(table, pd.DataFrame) else pd.DataFrame(table)
        display(table)
        os.makedirs(f'{DRIVE}/results/s0', exist_ok=True)
        table.to_csv(f'{DRIVE}/results/s0/structure_stats.csv', index=False)
        print('saved', f'{DRIVE}/results/s0/structure_stats.csv')
except Exception as e:
    print(f'[warn] structure stats unavailable: {e!r}')